# 00 - Overview: What is Spatial Transcriptomics?

Welcome. This series teaches **spatial transcriptomics (ST)** to someone who already
thinks in **images, voxels, segmentation, and CNN features** but has not worked with
**gene-expression data**. We will lean on that imaging intuition the whole way.

## Learning objectives
By the end of this chapter you will be able to:
1. Explain what spatial transcriptomics measures and why it exists.
2. Compare **bulk RNA-seq**, **single-cell RNA-seq (scRNA-seq)**, and **spatial transcriptomics**.
3. Describe the **10x Visium** assay and its four data objects.
4. Explain why **histology context** matters.
5. Connect ST to concrete **pharma / drug-discovery** use cases.


## 1. The one-sentence idea

> **Spatial transcriptomics measures how strongly thousands of genes are switched on,
> at many locations across an intact tissue slice, while keeping track of *where* each
> measurement came from on the matching histology image.**

Genes are the instructions; **RNA** molecules (transcripts) are the working copies a
cell makes when it 'uses' a gene. Counting RNA per gene tells you which programs the
tissue is running. Doing it *with spatial coordinates* means you can lay those programs
back onto the H&E image, like a functional overlay on an anatomical scan.

### Imaging analogy
Think of a CT volume where each voxel, instead of a single Hounsfield number, carried a
**~20,000-dimensional vector** of gene activity. ST is close to that: each location has a
high-dimensional molecular fingerprint *plus* a position on the tissue image.


## 2. Bulk vs single-cell vs spatial

| Property | Bulk RNA-seq | Single-cell RNA-seq | Spatial transcriptomics (Visium) |
|---|---|---|---|
| Unit measured | whole tissue (one average) | individual dissociated cells | spots on a grid (~1-10 cells each) |
| Spatial info | none | **lost** (tissue is dissociated) | **preserved** (xy on the image) |
| Single-cell purity | no | yes | no (each spot mixes cells) |
| Histology pairing | no | no | **yes** (paired H&E) |
| Typical use | average expression / DE | cell-type atlases | where programs live in tissue |

- **Bulk** is like one global histogram of the whole organ: cheap, but you lose all
  location and cellular detail.
- **scRNA-seq** is like perfectly segmenting every cell and profiling it - but you had to
  grind the tissue into a suspension first, so you **threw away the map**.
- **Spatial** keeps the map. The trade-off (for Visium) is resolution: a spot is a small
  neighborhood of cells, not one cell.


## 3. The Visium assay in one picture

A thin tissue section is placed on a slide printed with a grid of ~5,000 **spots**
(~55 um diameter, hexagonally packed). The slide is stained (**H&E**) and photographed,
then mRNA released from the tissue is captured **at each spot**. Every captured molecule
gets a **spot barcode** (which spot) and a **UMI** (which molecule), so after sequencing
you can build a **spots x genes count matrix**.

```mermaid
flowchart LR
    tissue[Tissue section on Visium slide] --> stain[H&E stain + microscope image]
    tissue --> capture[Capture mRNA at each spot]
    capture --> barcode[Spot barcode + UMI per molecule]
    barcode --> seq[Sequencing]
    seq --> matrix[Spots x genes count matrix]
    stain --> image[H&E image + scale factors]
    matrix --> reg[Registration via spot pixel coordinates]
    image --> reg
    reg --> anndata[AnnData: matrix + coords + image]
```

**A Visium spot is NOT a single cell.** It is a tiny *mini-bulk* of whatever cells fall
under that 55 um circle. Keep repeating this to yourself - it shapes every downstream
interpretation.


## 4. The four data objects (and how they register)

Every Visium dataset is really four linked pieces. Confusing them is the #1 beginner
mistake, so we name them precisely now and revisit them in notebook 03.

| Object | What it is | Where it lives (AnnData) |
|---|---|---|
| **Tissue image pixels** | the H&E photograph (RGB array) | `adata.uns['spatial'][lib]['images']` |
| **Visium spots** | capture locations (rows of the matrix) | `adata.obs` (one row per spot) |
| **Gene expression count matrix** | spots x genes UMI counts | `adata.X` |
| **Spatial coordinates** | each spot's pixel (x, y) | `adata.obsm['spatial']` |
| **Tissue scale factors** | px conversions between image resolutions / spot size | `adata.uns['spatial'][lib]['scalefactors']` |

**Image-to-expression registration** = the spot coordinates are stored in *full-resolution*
image pixels. To draw a spot on the downscaled `hires`/`lowres` image you multiply its
coordinate by the matching scale factor. That is the entire 'registration' - a scalar
multiply - because 10x already aligned the grid to the image for you.

### CT analogy table
| Medical imaging | Spatial transcriptomics |
|---|---|
| Voxel | Spot |
| Hounsfield intensity | UMI count (per gene) |
| One scalar per voxel | ~20,000 gene values per spot |
| DICOM volume + header | AnnData (matrix + coords + image + scalefactors) |
| Image registration / resampling | scale-factor multiply (spots <-> image) |
| Radiomics features | handcrafted per-spot histology features (notebook 09) |


## 5. Why histology context matters

Expression alone tells you *what* programs are active; the **H&E image** tells you *where*
and *in what morphological structure*. Pathologists read tissue architecture (glands,
tumor nests, immune infiltrate, fibrosis, necrosis) directly from H&E. ST lets you ask:
**does this morphology correspond to a particular molecular state?** That bridge -
morphology <-> molecules - is exactly what makes ST powerful, and it is what the second
half of this tutorial (notebooks 09-10) operationalizes.


## 6. Why pharma cares (drug discovery & development)

- **Target discovery** - find genes/pathways switched on specifically in the diseased
  niche (e.g. the tumor-stroma boundary), not just averaged over the whole biopsy.
- **Tumor microenvironment (TME) profiling** - map how tumor, immune, and stromal
  compartments are arranged; spatial organization predicts therapy response.
- **Biomarker discovery** - identify spatial expression patterns that stratify responders
  vs non-responders, beyond bulk averages.
- **Drug response mechanisms** - compare treated vs untreated tissue to see *where* a drug
  changes expression (pharmacodynamics in situ).
- **Patient stratification** - spatial signatures can define subtypes that bulk assays miss.
- **Toxicity & tissue pathology** - localize injury programs (e.g. hepatocyte stress) to
  specific zones in safety/tox studies.
- **Translational pathology** - connect routine H&E (cheap, ubiquitous in the clinic) to
  molecular state, enabling histology-based prediction models.

We return to these in notebook 12 with concrete extension ideas.


## Expected outputs
This chapter is conceptual - there is no code to run. You should now be able to define
*spot, UMI, count matrix, spatial coordinates, scale factors,* and *registration*, and to
explain why a spot is not a cell.

## Common pitfalls
- Treating a spot as a single cell (it is a small mixture).
- Confusing 'genes' with 'RNA counts' - we measure transcript abundance, a proxy for how
  much a gene is being used.
- Assuming spatial = single-cell resolution. Visium is multi-cell per spot.

## Interpretation
ST is a **map of molecular activity** registered to a histology image. Everything we do
next is either (a) cleaning/representing that map, or (b) relating it to the image.

## What this means biologically
Tissues are organized: cell types sit in structured neighborhoods, and that organization
drives function and disease. Preserving location while measuring expression lets us study
tissue the way it actually works - in place.

---
**Next:** `01_environment_setup.ipynb` - install the toolkit and meet the AnnData object.
